## Feature Engineering

In [ ]:
# Extract temporal features from Start_Time
df_clean['Hour'] = df_clean['Start_Time'].dt.hour
df_clean['Day_of_Week'] = df_clean['Start_Time'].dt.day_name()
df_clean['Day_of_Week_Num'] = df_clean['Start_Time'].dt.dayofweek  # Monday=0, Sunday=6
df_clean['Month'] = df_clean['Start_Time'].dt.month
df_clean['Month_Name'] = df_clean['Start_Time'].dt.month_name()
df_clean['Day'] = df_clean['Start_Time'].dt.day

# Create season feature
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_clean['Season'] = df_clean['Month'].apply(get_season)

# Create time of day categories
def get_time_period(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_clean['Time_Period'] = df_clean['Hour'].apply(get_time_period)

# Create weekend flag
df_clean['Is_Weekend'] = df_clean['Day_of_Week_Num'].isin([5, 6]).astype(int)




In [ ]:



# Make a copy to preserve original
df_features = df_clean.copy()

# 1. Rush Hour Flag (binary: 1 if rush hour, 0 otherwise)
# Based on EDA: peaks at 7-8 AM and 3-5 PM
df_features['Rush_Hour'] = df_features['Hour'].apply(
    lambda x: 1 if x in [7, 8, 16, 17] else 0
)

# 5. High Risk Junction Flag (Junction without traffic control)
df_features['High_Risk_Junction'] = (
    (df_features['Junction'] == True) & 
    (df_features['Traffic_Signal'] == False)
).astype(int)

print(f"Rush_Hour distribution:\n{df_features['Rush_Hour'].value_counts()}")
print(f"High_Risk_Junction distribution:\n{df_features['High_Risk_Junction'].value_counts()}")

STEP 1: FEATURE ENGINEERING
Rush_Hour distribution:
Rush_Hour
0    309606
1    135467
Name: count, dtype: int64
High_Risk_Junction distribution:
High_Risk_Junction
0    412343
1     32730
Name: count, dtype: int64


In [17]:
print("\n" + "="*80)
print("STEP 2: FEATURE SELECTION")
print("="*80)

# Columns to drop (not needed for modeling)
columns_to_drop = [
    # Original datetime columns (already extracted features from them)
    'Start_Time', 'End_Time', 'Weather_Timestamp',
    
    # Metadata columns (not predictive)
    'ID', 'Source', 'Description',
    
    # Geographic text fields (will use State, drop City due to high cardinality)
    'Street', 'City', 'County', 'Zipcode', 'Country',
    
    # Airport code (not useful)
    'Airport_Code',
    
    # Timezone (redundant with geographic location)
    'Timezone',
    
    # Twilight features (redundant with Hour and Sunrise_Sunset)
    'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight',
    
    # Start/End coordinates (keep as is, don't drop - useful for modeling)
    # 'Start_Lat', 'Start_Lng'  # KEEP these

    'Month_Name',  # Drop Month_Name since we have Month numeric and Season categorical,
    'Day_of_Week'  # Drop Day_of_Week since we have Day_of_Week_Num and Is_Weekend
]

# Drop columns
df_features = df_features.drop(columns=columns_to_drop)

print(f"\nColumns dropped: {len(columns_to_drop)}")
print(f"Remaining columns: {df_features.shape[1]}")
print(f"\nRemaining feature list:")
print(df_features.columns.tolist())


STEP 2: FEATURE SELECTION

Columns dropped: 18
Remaining columns: 36

Remaining feature list:
['Severity', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'State', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Hour', 'Day_of_Week_Num', 'Month', 'Day', 'Season', 'Time_Period', 'Is_Weekend', 'Rush_Hour', 'High_Risk_Junction']


In [18]:
#ENCODING CATEGORICAL VARIABLES
# Separate target variable before encoding
y = df_features['Severity']
X = df_features.drop(columns=['Severity'])

print(f"Target variable (y) shape: {y.shape}")
print(f"Features (X) shape before encoding: {X.shape}")

# Identify categorical columns
categorical_columns = X.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"\nCategorical columns to encode: {categorical_columns}")

# 1. Weather_Condition - One-hot encode
# Get top 10 weather conditions, group rest as "Other"
top_weather = X['Weather_Condition'].value_counts().head(10).index.tolist()
X['Weather_Condition'] = X['Weather_Condition'].apply(
    lambda x: x if x in top_weather else 'Other'
)

# 2. State - One-hot encode
# Get top 15 states, group rest as "Other"
top_states = X['State'].value_counts().head(15).index.tolist()
X['State'] = X['State'].apply(
    lambda x: x if x in top_states else 'Other'
)

# 3. Wind_Direction - One-hot encode (already standardized)

# 4. Sunrise_Sunset - Binary encode (Day=1, Night=0)
X['Sunrise_Sunset'] = X['Sunrise_Sunset'].map({'Day': 1, 'Night': 0})

# Apply one-hot encoding
categorical_to_encode = ['Weather_Condition', 'State', 'Wind_Direction', 'Season','Time_Period']

X_encoded = pd.get_dummies(X, columns=categorical_to_encode, drop_first=True, dtype=int)

print(f"\nFeatures (X) shape after encoding: {X_encoded.shape}")
print(f"New feature count: {X_encoded.shape[1]}")



Target variable (y) shape: (445073,)
Features (X) shape before encoding: (445073, 35)

Categorical columns to encode: ['State', 'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset', 'Season', 'Time_Period']

Features (X) shape after encoding: (445073, 78)
New feature count: 78
